In [1]:
import duckdb
import pandas as pd
import unicodedata
from collections import Counter
import re
from nltk.corpus import stopwords
import string

In [2]:
from pathlib import Path

In [57]:
conexion = duckdb.connect(
    r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\bases\info_cc.db"
    )

centros_comerciales = conexion.execute(
    "SELECT *" \
    " From centros_comerciales " 
    ).df()

In [58]:
centros_comerciales

,centro_comercial,provincia,canton,parroquia,perimetro_calles,diccionarios_contexto
0,Recreo,pichincha,quito,magdalena,"antonio jose de sucre,av de la prensa,john ken...","[{""palabra"":""maldonado"",""frecuencia"":179},{""pa..."
1,Condado Shopping,pichincha,quito,"ponceano,cotocollao","lauro guerrero becerra,oe1e,oe2f,pedro vicente...","[{""palabra"":""kennedy"",""frecuencia"":9},{""palabr..."
2,Mall de los Andes,tungurahua,ambato,huachi chico,"atahualpa,cosmopolita,marcos montalvo,victor hugo","[{""palabra"":""victor"",""frecuencia"":27},{""palabr..."
3,San Marino,guayas,guayaquil,tarqui,"10 no,francisco de orellana,2do pasaje 10 no","[{""palabra"":""orellana"",""frecuencia"":82},{""pala..."
4,Paseo Shopping,manabi,portoviejo,andres vera,"25 de junio,9 de agosto,america,metropolitana,...","[{""palabra"":""washington"",""frecuencia"":4},{""pal..."
5,Portal Shopping,pichincha,quito,"calderon,carapungo","giovanni calles lascano,norte,simon bolivar","[{""palabra"":""bolivar"",""frecuencia"":61},{""palab..."
6,Paseo San Francisco,pichincha,quito,cumbaya,"maria angelica idrobo,oswaldo guayasamin,oe4d,...","[{""palabra"":""interoceanica"",""frecuencia"":17},{..."
7,El Jardin,pichincha,quito,inaquito,"av de la republica,mariana de jesus,potosi,rio...","[{""palabra"":""amazonas"",""frecuencia"":23},{""pala..."
8,Quicentro,pichincha,quito,inaquito,"6 de diciembre,de los shyris,el comercio,nacio...","[{""palabra"":""naciones"",""frecuencia"":64},{""pala..."
9,Mall del Pacifico,manabi,manta,manta,"20,23,2da paralela,malecon","[{""palabra"":""calle"",""frecuencia"":27},{""palabra..."


In [6]:
centros = []
for centro in centros_comerciales:
    centros.append(centro[0])

In [8]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\bases\base_rucs_sri.parquet")

In [9]:
#Nos quedamos solo con los rucs interesantes

base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)

base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()]
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [10]:
#Normalizamos los q gramas
def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [11]:
def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def limpiador(lista: list):
    documentos = []
    re_punctuation = re.compile('[%s]' % re.escape(string.punctuation))

    stop_words_s = set(stopwords.words('spanish'))
    stop_words = set(stopwords.words('english'))

    for descripcion in lista:
        tokens = descripcion.split()
        tokens = [re_punctuation.sub(' ', w) for w in tokens]
        tokens = ' '.join(tokens).split()

        tokens = [quitar_tildes(word.lower()) for word in tokens]

        tokens = [word.lower() for word in tokens if re.search('[a-z ]', word.lower())]
        tokens = [word.lower() for word in tokens if word.isalpha()]
        
        tokens = [w for w in tokens if w not in stop_words_s]
        tokens = [w for w in tokens if w not in stop_words]
        
        tokens = [word for word in tokens if len(word) > 2]

        if len(tokens) > 0:
            documento = ' '.join(tokens)
        else:
            documento = ''  # ← asegura longitud igual al DF

        documentos.append(documento)

    return documentos

In [12]:
#Normalizamos el nombre fantasía comercial
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar)

#Tenemos la direccion completa separada en base registro civil
base_registro_civil[['provincia',  'canton', 'parroquia', 'calles']] = (
    base_registro_civil['direccion_completa']
        .str.split('/', n=3, expand=True)
)

In [13]:
mask_regex = base_registro_civil['nombre_fantasia_comercial'].str.contains(rf'(?=.*quicentro)', case = False, na = False)
mask_canton = base_registro_civil['canton'].str.contains(f"quito", case = False, na =  False)
mask_parroquia =  base_registro_civil['parroquia'].str.contains(f"naciones", case = False, na =  False)


In [20]:
#Para sacarnos las frecuencias
def obtener_frecuencias(nombre: str, apellido: str, canton: str, parroquia: str, especifico: bool)->dict:
    #Nombre del centro comercial
    if nombre == "na":
        rejex = f'(?=.*{apellido})'
    else:
        rejex = f'(?=.*{nombre})(?=.*{apellido})'

    print(rejex)
    #Filtros para las calles
    mask_regex = base_registro_civil['nombre_fantasia_comercial'].str.contains(rf'{rejex}', case = False, na = False)
    mask_canton = base_registro_civil['canton'].str.contains(f"{canton}", case = False, na =  False)
    
    if especifico == True:
        mask_parroquia =  base_registro_civil['parroquia'].str.contains(f"{parroquia}", case = False, na =  False)
        mask_canton = mask_canton & mask_parroquia

    #Filtramos la base para obtener las calles
    base_filtrada = base_registro_civil[mask_regex & mask_canton]
    print(base_filtrada[['nombre_fantasia_comercial','calles']].head())
    #Obtenemos las calles
    list_calles =list(base_filtrada['calles'])

    #Limpiamos las calles y hacemos una sola list
    list_calles_limpia = limpiador(list_calles)
    list_calles_limpia_total = [
        palabra
        for i in range(len(list_calles_limpia))
        for palabra in list_calles_limpia[i].split()
    ]

    #Sacamos las frecuencias
    frecuencias = Counter(list_calles_limpia_total)

    #Hacemos dataframe de frecuencias
    df_frecuencias = ( 
        pd.DataFrame(frecuencias.items(), columns = ['palabra', 'frecuencia'])
        .sort_values('frecuencia', ascending = False)
        .reset_index(drop = True)
    )

    return df_frecuencias


In [91]:
nombre = "na"
apellido = "quicentro"
canton = "quito"
parroquia = "iñaquito"
especifico = True
df_frecuencias = obtener_frecuencias(nombre, apellido, canton, parroquia, especifico)
#Guardamos las frecuencias
df_frecuencias.to_excel(rf"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\busqueda_rucs\frecuencias_{apellido}.xlsx", index = False)
print(f"se guardó frecuencias_{apellido}.xlsx")

(?=.*quicentro)
        nombre_fantasia_comercial  \
321743       isla quicentro norte   
343456          jomatik quicentro   
966664            stone quicentro   
3296201     pet station quicentro   
3296813    ferrisariato quicentro   

                                               calles  
321743    AV. 6 DE DICIEMBRE SN Y AV. NACIONES UNIDAS  
343456      AV NACIONES UNIDAS SN Y AV 6 DE DICIEMBRE  
966664                   NACIONES UNIDAS S/N Y SHYRIS  
3296201        EL COMERCIO N37B Y CALLE 9 EL VENGADOR  
3296813   AV. NACIONES UNIDAS S/N Y REP. DEL SALVADOR  
se guardó frecuencias_quicentro.xlsx


In [95]:
df_frecuencias.columns

Index(['palabra', 'frecuencia'], dtype='object')

In [14]:
conexion.execute("CREATE TABLE centros_comerciales_temporal AS" \
" (SELECT * FROM centros_comerciales)")

In [18]:
conexion.execute("ALTER TABLE centros_comerciales " \
"ADD COLUMN diccionarios_contexto JSON")

In [15]:
direccion_standar = Path(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\busqueda_rucs")
diccionario = dict()
for ruta in direccion_standar.glob("frecuencias_*.xlsx"):
     diccionario[str(ruta.stem).replace("frecuencias_", "")]= pd.read_excel(ruta).to_dict(orient="records")

In [16]:
diccionario.keys()

dict_keys(['andes', 'bosque', 'condado', 'francisco', 'inaquito', 'jardin', 'marino', 'pacifico', 'paseo_portoviejo', 'portal', 'quicentro', 'recreo', 'scala', 'sol'])

In [53]:
centro = "sol"
dicc = diccionario[centro]

print(conexion.execute(
"UPDATE centros_comerciales " \
f"SET diccionarios_contexto = {dicc} " \
f"WHERE centro_comercial ILIKE '%{centro}%'"
).df())

print(
  conexion.execute(
    "Select diccionarios_contexto, centro_comercial " \
    "from centros_comerciales" \
    f" where centro_comercial ILIKE '%{centro}%'"
).df()  
)

   Count
0      1
                               diccionarios_contexto centro_comercial
0  [{"palabra":"marengo","frecuencia":92},{"palab...     Mall del Sol


In [54]:
conexion.close()